# Tech Challenge Fase 3 - Analise Exploratoria de Dados

**Objetivo:** entender a camada Gold da Fase 2, validar a formulacao do problema supervisionado e produzir evidencias para a modelagem.

**Fonte:** dados Parquet em `../data/gold/`.

**Observacao analitica:** a base disponivel esta em granularidade agregada por municipio, ano, serie e rede. Portanto, este projeto modela risco/cumprimento de meta em nivel municipal/agregado, nao alfabetizacao individual por aluno.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='deep')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

DATA_DIR = Path('../data/gold')
IMAGE_DIR = Path('../images')
IMAGE_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(name: str):
    path = IMAGE_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches='tight')
    print(f'Figura salva em: {path}')

## 1. Carregamento Dos Dados

A camada Gold possui seis tabelas analiticas. A tabela `indicador_municipio` e a base principal para a modelagem por ter a menor granularidade disponivel no projeto.

In [ ]:
tables = {
    'indicador_municipio': DATA_DIR / 'indicador_municipio/indicador_municipio.parquet',
    'proficiencia_municipio': DATA_DIR / 'proficiencia_municipio/proficiencia_municipio.parquet',
    'evolucao_temporal_uf': DATA_DIR / 'evolucao_temporal_uf/evolucao_temporal_uf.parquet',
    'ranking_uf': DATA_DIR / 'ranking_uf/ranking_uf.parquet',
    'municipios_risco': DATA_DIR / 'municipios_risco/municipios_risco.parquet',
    'comparativo_nacional': DATA_DIR / 'comparativo_nacional/comparativo_nacional.parquet',
}

datasets = {name: pd.read_parquet(path) for name, path in tables.items()}
indicador = datasets['indicador_municipio'].copy()
proficiencia = datasets['proficiencia_municipio'].copy()
evolucao_uf = datasets['evolucao_temporal_uf'].copy()
ranking_uf = datasets['ranking_uf'].copy()
risco = datasets['municipios_risco'].copy()
comparativo = datasets['comparativo_nacional'].copy()

summary = pd.DataFrame([
    {'tabela': name, 'linhas': df.shape[0], 'colunas': df.shape[1]}
    for name, df in datasets.items()
]).sort_values('linhas', ascending=False)
summary

## 2. Tabela Principal - `indicador_municipio`

A tabela principal contem 23.995 registros e 18 colunas, organizados por municipio, ano, serie e rede.

In [ ]:
indicador.head()

In [ ]:
pd.DataFrame({
    'coluna': indicador.columns,
    'tipo': indicador.dtypes.astype(str).values,
    'valores_unicos': [indicador[c].nunique(dropna=False) for c in indicador.columns],
    'nulos': indicador.isna().sum().values,
    'perc_nulos': (indicador.isna().mean().values * 100).round(2),
})

In [ ]:
indicador.describe(include='all').transpose()

### Leitura inicial

- Existem dados para 2023 e 2024.
- A serie disponivel na base Gold e a serie 2.
- A coluna `rede` esta codificada numericamente; como nao ha dicionario oficial no repositorio, ela sera tratada como categoria codificada.
- As metas de 2030 estao constantes em 80 nesta versao da base, entao tendem a ter pouco poder preditivo isolado.

In [ ]:
print('Registros por ano:')
print(indicador['ano'].value_counts().sort_index())

print('')
print('Registros por serie:')
print(indicador['serie'].value_counts(dropna=False).sort_index())

print('')
print('Registros por rede codificada:')
print(indicador['rede'].value_counts(dropna=False).sort_index())

print('')
print('Municipios unicos:', indicador['id_municipio'].nunique())
print('UFs unicas:', indicador['sigla_uf'].nunique())

## 3. Valores Nulos

A principal concentracao de nulos esta nas colunas de proporcao por nivel. Pela documentacao inicial, essas proporcoes nao estao disponiveis em 2023.

In [ ]:
nulls = (
    indicador.isna().sum()
    .rename('nulos')
    .to_frame()
    .assign(perc=lambda x: (x['nulos'] / len(indicador) * 100).round(2))
    .query('nulos > 0')
    .sort_values('nulos', ascending=False)
)
nulls

In [ ]:
nulls_by_year = indicador.groupby('ano').apply(lambda x: x.isna().sum(), include_groups=False)
nulls_by_year.loc[:, (nulls_by_year > 0).any(axis=0)]

In [ ]:
null_plot = nulls.reset_index().rename(columns={'index': 'coluna'})
plt.figure(figsize=(11, 5))
sns.barplot(data=null_plot, y='coluna', x='perc', color='#4C78A8')
plt.title('Percentual de valores nulos por coluna')
plt.xlabel('% de nulos')
plt.ylabel('')
save_fig('eda_nulos_por_coluna.png')
plt.show()

### Decisao para modelagem

As colunas `proporcao_aluno_nivel_*` podem enriquecer a leitura educacional, mas possuem nulos em todos os registros de 2023. Para o primeiro modelo, elas devem ser avaliadas em cenarios separados: usar somente 2024, imputar valores ou remove-las do conjunto principal de features.

## 4. Variavel Target - `categoria_risco`

O desafio pede classificacao binaria. Como a base atual esta agregada, `categoria_risco` sera usada como proxy para risco/cumprimento de meta municipal.

**Mapeamento:** `meta_atingida` = 1; `critico`, `alto` e `moderado` = 0.

In [ ]:
indicador['target'] = (indicador['categoria_risco'] == 'meta_atingida').astype(int)

target_counts = (
    indicador['target']
    .value_counts()
    .sort_index()
    .rename(index={0: '0_em_risco', 1: '1_meta_atingida'})
    .to_frame('registros')
)
target_counts['percentual'] = (target_counts['registros'] / target_counts['registros'].sum() * 100).round(2)
target_counts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

categoria_order = ['critico', 'alto', 'moderado', 'meta_atingida']
sns.countplot(data=indicador, x='categoria_risco', order=categoria_order, ax=axes[0], color='#4C78A8')
axes[0].set_title('Distribuicao de categoria_risco')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=25)

sns.countplot(data=indicador, x='target', ax=axes[1], color='#F58518')
axes[1].set_title('Distribuicao do target binario')
axes[1].set_xlabel('0 = em risco | 1 = meta atingida')
axes[1].set_ylabel('Registros')

save_fig('eda_distribuicao_target.png')
plt.show()

In [ ]:
target_by_year = pd.crosstab(
    indicador['ano'],
    indicador['target'],
    normalize='index'
).rename(columns={0: 'em_risco', 1: 'meta_atingida'}).round(4) * 100

target_by_year

### Leitura do target

A classe positiva representa cerca de 19% da base. Isso indica desbalanceamento moderado e reforca a necessidade de acompanhar `precision`, `recall`, `F1-score` e matriz de confusao, nao apenas acuracia.

## 5. Distribuicoes Numericas

In [ ]:
num_cols = [
    'taxa_alfabetizacao', 'media_portugues', 'meta_mun_2030',
    'meta_uf_2030', 'meta_brasil_2030', 'gap_meta_municipio_2030',
    'gap_meta_uf_2030', 'proporcao_aluno_nivel_0',
    'proporcao_aluno_nivel_1', 'proporcao_aluno_nivel_2',
    'proporcao_aluno_nivel_3'
]

indicador[num_cols].hist(bins=35, figsize=(15, 11), color='#4C78A8')
plt.suptitle('Distribuicao das variaveis numericas', y=1.02)
save_fig('eda_distribuicoes_numericas.png')
plt.show()

In [ ]:
plt.figure(figsize=(11, 5))
sns.boxplot(data=indicador, x='categoria_risco', y='taxa_alfabetizacao', order=categoria_order, color='#72B7B2')
plt.title('Taxa de alfabetizacao por categoria de risco')
plt.xlabel('Categoria de risco')
plt.ylabel('Taxa de alfabetizacao')
save_fig('eda_taxa_por_categoria_risco.png')
plt.show()

### Leitura das distribuicoes

A taxa de alfabetizacao varia bastante entre os registros. Como `categoria_risco` e derivada do desempenho em relacao a meta, a separacao entre categorias aparece claramente no boxplot. Essa relacao e util para explicar o problema, mas confirma que taxa e gaps nao devem entrar como features no modelo preditivo principal.

## 6. Correlacoes

A matriz abaixo inclui variaveis numericas para apoiar a interpretacao exploratoria. Variaveis com leakage aparecem aqui apenas para diagnostico, nao como recomendacao de uso no modelo principal.

In [ ]:
corr_cols = [c for c in num_cols + ['target'] if c in indicador.columns]
corr = indicador[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', center=0, square=False)
plt.title('Matriz de correlacao - variaveis numericas')
save_fig('eda_correlacao_variaveis.png')
plt.show()

In [ ]:
corr['target'].drop('target').sort_values(key=lambda x: x.abs(), ascending=False).to_frame('correlacao_com_target')

## 7. Analise Por UF, Ano, Rede E Serie

In [ ]:
uf_summary = (
    indicador.groupby('sigla_uf')
    .agg(
        taxa_media=('taxa_alfabetizacao', 'mean'),
        media_portugues=('media_portugues', 'mean'),
        registros=('id_municipio', 'count'),
        municipios=('id_municipio', 'nunique'),
        perc_meta_atingida=('target', 'mean'),
    )
    .assign(perc_meta_atingida=lambda x: (x['perc_meta_atingida'] * 100).round(2))
    .round({'taxa_media': 2, 'media_portugues': 2})
    .sort_values('taxa_media', ascending=False)
)
uf_summary.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)
uf_plot = uf_summary.sort_values('taxa_media')
sns.barplot(data=uf_plot.reset_index(), y='sigla_uf', x='taxa_media', ax=axes[0], color='#4C78A8')
axes[0].axvline(80, color='#E45756', linestyle='--', linewidth=1.5, label='Meta 2030')
axes[0].set_title('Taxa media de alfabetizacao por UF')
axes[0].set_xlabel('Taxa media')
axes[0].set_ylabel('UF')
axes[0].legend()

sns.barplot(data=uf_plot.reset_index(), y='sigla_uf', x='perc_meta_atingida', ax=axes[1], color='#F58518')
axes[1].set_title('% de registros com meta atingida por UF')
axes[1].set_xlabel('% meta atingida')
axes[1].set_ylabel('')

save_fig('eda_taxa_e_target_por_uf.png')
plt.show()

In [ ]:
year_summary = (
    indicador.groupby('ano')
    .agg(
        registros=('id_municipio', 'count'),
        municipios=('id_municipio', 'nunique'),
        taxa_media=('taxa_alfabetizacao', 'mean'),
        media_portugues=('media_portugues', 'mean'),
        perc_meta_atingida=('target', 'mean'),
    )
    .assign(perc_meta_atingida=lambda x: (x['perc_meta_atingida'] * 100).round(2))
    .round({'taxa_media': 2, 'media_portugues': 2})
)
year_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=year_summary.reset_index(), x='ano', y='taxa_media', ax=axes[0], color='#4C78A8')
axes[0].axhline(80, color='#E45756', linestyle='--', linewidth=1.5)
axes[0].set_title('Taxa media por ano')
axes[0].set_xlabel('Ano')
axes[0].set_ylabel('Taxa media')

sns.barplot(data=year_summary.reset_index(), x='ano', y='perc_meta_atingida', ax=axes[1], color='#F58518')
axes[1].set_title('% meta atingida por ano')
axes[1].set_xlabel('Ano')
axes[1].set_ylabel('% meta atingida')

save_fig('eda_taxa_por_ano.png')
plt.show()

In [ ]:
rede_summary = (
    indicador.groupby('rede')
    .agg(
        registros=('id_municipio', 'count'),
        taxa_media=('taxa_alfabetizacao', 'mean'),
        media_portugues=('media_portugues', 'mean'),
        perc_meta_atingida=('target', 'mean'),
    )
    .assign(perc_meta_atingida=lambda x: (x['perc_meta_atingida'] * 100).round(2))
    .round({'taxa_media': 2, 'media_portugues': 2})
    .sort_index()
)
rede_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=rede_summary.reset_index(), x='rede', y='taxa_media', ax=axes[0], color='#4C78A8')
axes[0].axhline(80, color='#E45756', linestyle='--', linewidth=1.5)
axes[0].set_title('Taxa media por rede codificada')
axes[0].set_xlabel('Rede')
axes[0].set_ylabel('Taxa media')

sns.barplot(data=rede_summary.reset_index(), x='rede', y='perc_meta_atingida', ax=axes[1], color='#F58518')
axes[1].set_title('% meta atingida por rede codificada')
axes[1].set_xlabel('Rede')
axes[1].set_ylabel('% meta atingida')

save_fig('eda_taxa_por_rede.png')
plt.show()

In [ ]:
serie_summary = (
    indicador.groupby('serie')
    .agg(
        registros=('id_municipio', 'count'),
        taxa_media=('taxa_alfabetizacao', 'mean'),
        perc_meta_atingida=('target', 'mean'),
    )
    .assign(perc_meta_atingida=lambda x: (x['perc_meta_atingida'] * 100).round(2))
    .round({'taxa_media': 2})
)
serie_summary

### Leitura por dimensoes

- As UFs apresentam diferencas relevantes na taxa media e no percentual de registros com meta atingida.
- A taxa media melhora levemente de 2023 para 2024, e o percentual de meta atingida tambem aumenta.
- A base contem apenas a serie 2, entao `serie` nao agrega variacao para o primeiro modelo.
- `rede` mostra diferencas entre grupos, mas precisa ser mantida como codigo ate termos um dicionario de dados confiavel.

## 8. Municipios Em Maior Risco

A tabela `municipios_risco` traz os 100 municipios com pior gap em relacao a meta da UF no ano mais recente.

In [ ]:
risco.head(20)

In [ ]:
plt.figure(figsize=(12, 7))
top_risco = risco.head(20).copy()
top_risco['municipio_uf'] = top_risco['id_municipio'] + ' - ' + top_risco['sigla_uf']
sns.barplot(data=top_risco.sort_values('gap_meta_uf_2030'), y='municipio_uf', x='gap_meta_uf_2030', color='#E45756')
plt.title('Top 20 municipios com maior distancia negativa da meta')
plt.xlabel('Gap em relacao a meta da UF')
plt.ylabel('Municipio - UF')
save_fig('eda_top20_municipios_risco.png')
plt.show()

In [ ]:
risco.groupby('sigla_uf').size().sort_values(ascending=False).to_frame('municipios_no_top100_risco').head(10)

## 9. Comparativo Nacional E Evolucao Temporal

In [ ]:
comparativo.head(10)

In [ ]:
plt.figure(figsize=(11, 5))
comp_publica = comparativo[comparativo['rede'].str.lower() == 'pública'].copy()
sns.lineplot(data=comp_publica, x='ano', y='taxa_alfabetizacao', marker='o', label='Taxa observada')
sns.lineplot(data=comp_publica, x='meta_ano', y='meta_valor', marker='o', label='Trajetoria de meta')
plt.title('Comparativo nacional: taxa observada vs trajetoria de meta')
plt.xlabel('Ano')
plt.ylabel('Percentual')
save_fig('eda_comparativo_nacional_meta.png')
plt.show()

In [ ]:
evolucao_uf.sort_values(['sigla_uf', 'ano']).head(10)

## 10. Hipoteses Analiticas

**H1 - Municipios com maior `media_portugues` tem maior probabilidade de atingir a meta.**

A correlacao e as distribuicoes indicam associacao positiva entre proficiencia media em portugues e cumprimento da meta. Essa sera uma feature importante para a modelagem.

**H2 - Gaps em relacao as metas indicam risco educacional.**

Confirmada para analise explicativa. Entretanto, `gap_meta_*` e derivado da propria taxa de alfabetizacao e sera excluido do modelo preditivo principal por data leakage.

**H3 - Existem padroes regionais claros.**

As diferencas entre UFs indicam padroes territoriais relevantes. Como a base atual possui `sigla_uf`, essa variavel deve entrar como feature categorica no primeiro modelo.

**H4 - Rede influencia o desempenho observado.**

A EDA por rede codificada mostra diferencas entre grupos. A variavel sera considerada como categorica, mas sua interpretacao de negocio depende de um dicionario oficial de codigos.

## 11. Decisoes Para A Modelagem

- **Tipo de problema:** classificacao supervisionada binaria.
- **Granularidade:** municipio x ano x serie x rede.
- **Target:** `categoria_risco` -> binario (`meta_atingida` = 1, demais categorias = 0).
- **Desbalanceamento:** classe positiva em torno de 19%; avaliar `class_weight='balanced'` e metricas alem de acuracia.
- **Features candidatas sem leakage:** `ano`, `sigla_uf`, `rede`, `media_portugues`, `meta_mun_2030`, `meta_uf_2030`, `meta_brasil_2030` e, em cenario separado, `proporcao_aluno_nivel_*`.
- **Features a excluir do modelo preditivo principal por leakage:** `taxa_alfabetizacao`, `gap_meta_municipio_2030`, `gap_meta_uf_2030`, `atingiu_meta_uf` e `categoria_risco`.
- **Serie:** como a base atual possui apenas a serie 2, a coluna pode ser removida do primeiro modelo por nao trazer variacao.
- **Validacao inicial:** usar split estratificado; avaliar tambem treino em 2023 e teste em 2024 para observar generalizacao temporal.

## 12. Graficos Gerados

Os principais graficos desta EDA foram salvos em `../images/`:

- `eda_nulos_por_coluna.png`
- `eda_distribuicao_target.png`
- `eda_distribuicoes_numericas.png`
- `eda_taxa_por_categoria_risco.png`
- `eda_correlacao_variaveis.png`
- `eda_taxa_e_target_por_uf.png`
- `eda_taxa_por_ano.png`
- `eda_taxa_por_rede.png`
- `eda_top20_municipios_risco.png`
- `eda_comparativo_nacional_meta.png`